# Adaptation Workflow Test

This notebook does three things:

1. Runs the legacy protection workflow on real flood-risk data
2. Runs the new scenario-builder protection workflow on the same data
3. Compares outputs so the two approaches can be checked side by side

It also includes setup templates for vulnerability and frequency adaptation.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import numpy as np
import pandas as pd

from sovereign.flood import (
    AdaptationSpec,
    BasinLossCurve,
    apply_basin_frequency_shift,
    build_basin_curves,
    build_frequency_scenario_curves,
    build_protection_scenario_curves,
    build_vulnerability_scenario_curves,
    extract_sectoral_losses,
    run_simulation,
)


In [2]:
# USER CONFIG
model = "wri"
n_years = 5000
adaptation_aep = 0.01  # 100-year protection

root = Path.cwd().parent
risk_basin_path = os.path.join(root, "outputs", "flood", "risk", "basins", f"risk_basins_m-{model}.csv")
copula_path = os.path.join(root, "outputs", "flood", "dependence", "copulas", "copula_random_numbers.gzip")


In [3]:
# Load baseline risk data
risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:]
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

copula_random_numbers = pd.read_parquet(copula_path)
copula_random_numbers = copula_random_numbers.iloc[:n_years].copy()

risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


## Protection Comparison

The comparison below uses the real legacy protection inputs already embedded in
the basin risk table: baseline damages plus raster-derived `adapted_damages`.


In [4]:
# Build baseline curves once
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(risk_data)

# Legacy protection workflow
legacy_baseline_losses, legacy_adapted_losses = run_simulation(
    baseline_curves,
    n_years,
    adaptation_aep,
    copula_random_numbers,
)


C:\Users\Mark.DESKTOP-UFHIN6T\AppData\Local\Temp\ipykernel_29844\1372039298.py:5: DeprecationWarning: The legacy run_simulation(basin_curves, n_years, adaptation_aep, copula_numbers) signature is deprecated. Prefer passing baseline_curves and scenario_curves directly.
  legacy_baseline_losses, legacy_adapted_losses = run_simulation(
100%|█████████████████████████████████████████████████████████████████████████████| 5000/5000 [00:44<00:00, 113.22it/s]


In [5]:
# New protection workflow using the explicit raster-derived scenario builder
adapted_risk_data = risk_data.copy()
adapted_risk_data["damages"] = adapted_risk_data["adapted_damages"]

protection_scenario_curves = build_protection_scenario_curves(
    baseline_risk_df=risk_data,
    adapted_risk_df=adapted_risk_data,
    adapted_protection_aep=adaptation_aep,
)

new_baseline_losses, new_adapted_losses = run_simulation(
    baseline_curves,
    protection_scenario_curves,
    n_years,
    copula_random_numbers,
)


C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\sovereign-risk\lib\site-packages\sovereign\flood.py:515: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scenario_df = pd.concat([baseline_window, adapted_window], ignore_index=True)
100%|█████████████████████████████████████████████████████████████████████████████| 5000/5000 [00:44<00:00, 113.09it/s]


In [6]:
# Compare legacy and new outputs by sector
all_sectors = sorted(legacy_baseline_losses.keys())
comparison_rows = []

for sector in all_sectors:
    comparison_rows.append({
        "sector": sector,
        "baseline_max_abs_diff": float(np.max(np.abs(legacy_baseline_losses[sector] - new_baseline_losses[sector]))),
        "adapted_max_abs_diff": float(np.max(np.abs(legacy_adapted_losses[sector] - new_adapted_losses[sector]))),
        "baseline_mean_diff": float(np.mean(legacy_baseline_losses[sector] - new_baseline_losses[sector])),
        "adapted_mean_diff": float(np.mean(legacy_adapted_losses[sector] - new_adapted_losses[sector])),
        "baseline_allclose": bool(np.allclose(legacy_baseline_losses[sector], new_baseline_losses[sector])),
        "adapted_allclose": bool(np.allclose(legacy_adapted_losses[sector], new_adapted_losses[sector])),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,sector,baseline_max_abs_diff,adapted_max_abs_diff,baseline_mean_diff,adapted_mean_diff,baseline_allclose,adapted_allclose
0,Agriculture,0.0,0.0,0.0,0.0,True,True
1,Manufacturing,0.0,0.0,0.0,0.0,True,True
2,Private,0.0,0.0,0.0,0.0,True,True
3,Public,0.0,0.0,0.0,0.0,True,True
4,Service,0.0,0.0,0.0,0.0,True,True


In [7]:
# Compare aggregate loss metrics
legacy_sectoral = extract_sectoral_losses(legacy_adapted_losses, n_years)
new_sectoral = extract_sectoral_losses(new_adapted_losses, n_years)

aggregate_comparison = pd.DataFrame({
    "metric": ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"],
    "legacy_aal": [legacy_sectoral[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
    "new_aal": [new_sectoral[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
})
aggregate_comparison["aal_diff"] = aggregate_comparison["legacy_aal"] - aggregate_comparison["new_aal"]
aggregate_comparison


,metric,legacy_aal,new_aal,aal_diff
0,GVA_loss,2.241635e+07,2.241635e+07,0.0
1,CAP_dam,2.431491e+07,2.431491e+07,0.0
2,AGR_loss,1.451419e+07,1.451419e+07,0.0
3,MAN_loss,2.635769e+06,2.635769e+06,0.0
4,SER_loss,5.266391e+06,5.266391e+06,0.0
5,PUB_dam,8.572607e+06,8.572607e+06,0.0
6,PRI_dam,1.574230e+07,1.574230e+07,0.0


## Vulnerability Adaptation Template

For vulnerability adaptation the preferred workflow is:

- prepare an adaptation raster that flags adapted cells
- rerun raster damages for the targeted sector using a reduced vulnerability curve in those cells
- aggregate those adapted raster damages to a basin risk table
- pass that basin table into `build_vulnerability_scenario_curves(...)`


In [ ]:
# Template: replace this with a raster-derived adapted basin risk table
vulnerability_adapted_risk_data = risk_data.copy()
# vulnerability_adapted_risk_data["damages"] = ...  # replace with damages aggregated from adapted vulnerability rasters

# Example usage after adapted basin damages are prepared:
# vulnerability_curves = build_vulnerability_scenario_curves(risk_data, vulnerability_adapted_risk_data)
# vuln_baseline_losses, vuln_adapted_losses = run_simulation(baseline_curves, vulnerability_curves, n_years, copula_random_numbers)


## Frequency Adaptation Template

For frequency adaptation the preferred workflow is basin-level:

- specify which basins are affected
- provide a lookup table with either `RP` and `RP_future`, or `AEP` and `AEP_future`
- build shifted basin curves directly from the baseline risk table


In [ ]:
# Template lookup table: replace with your own NBS shift assumptions
frequency_lookup = pd.DataFrame({
    "RP": sorted(risk_data["RP"].unique()),
    "RP_future": sorted(risk_data["RP"].unique()),
})

# Example set of target basins
target_basins = sorted(risk_data["HB_L6"].unique())[:5]

# Example usage after the lookup and basin list are populated:
# shifted_risk_data = apply_basin_frequency_shift(risk_data, frequency_lookup, basin_ids=target_basins, degrade_protection=False)
# frequency_curves = build_frequency_scenario_curves(risk_data, frequency_lookup, basin_ids=target_basins, degrade_protection=False)
# freq_baseline_losses, freq_adapted_losses = run_simulation(baseline_curves, frequency_curves, n_years, copula_random_numbers)
